# East River station comparison with MB and RMB

This notebook combines `Point_Comp_p_SWE_bias.ipynb` and
`Point_Comp_f_density.ipynb` and produces **two separate figures** for
**WY2018–WY2026**, with columns ordered **Butte, Schofield Pass, Upper Taylor**.

1. **4 × 3 figure:** precipitation MB, precipitation RMB, SWE MB, SWE RMB.
2. **2 × 3 figure:** annual $f_{\mathrm{SWE,precip}}$ and annual-mean snow density.

## Statistical definitions

For each station, water year, variable, and model, daily model and SNOTEL
values are joined by date and only pairs with both values finite are retained.

\[
\mathrm{MB}=\frac{1}{n}\sum_{i=1}^n(M_i-O_i),\qquad
\mathrm{RMB}(\%)=100\frac{\sum_{i=1}^n(M_i-O_i)}{\sum_{i=1}^n O_i}.
\]

RMB is the ratio of the paired sums, **not** the mean of daily percentage
errors. Positive values indicate model overestimation. MB and RMB are NaN
when there are no finite pairs; RMB is also NaN when the paired SNOTEL sum
is zero. Zero-observation days otherwise remain in the paired sample.
Precipitation MB is in mm day$^{-1}$; SWE MB is in mm. No correlations are
calculated, imported as statistical helpers, plotted, or exported.

Precipitation uses SNOTEL `precip_gapfilled_mm`, iSnobal
`precip_nearest_mm`, and Noah-MP `precip_mm`. SWE uses SNOTEL `swe_mm`,
iSnobal `swe_nearest_mm`, and Noah-MP `swe_mm`. Each model uses its own
available matched dates. WY2026 retains the workflow's available partial-year record.

The original SWE MB consistency check against Table 4 is retained and now
also detects missing station-year records and mismatched NaNs. No metrics
CSV is written until that check passes.

## Snow fraction and density

The validated Table 4 $f_{\mathrm{SWE,precip}}$ fields are retained: source-specific
peak SWE divided by calendar-complete precipitation from 1 October through
that source's peak date. The workflow source implementation is checked before use.

Daily snow density is $1000\,\mathrm{SWE}_{mm}/\mathrm{SD}_{mm}$ in kg m$^{-3}$;
the annual result is the arithmetic mean of valid daily density for each source.
**Confirmed settings:** finite SWE and depth, **snow depth ≥100 mm**, and
**density axis 200–400 kg m$^{-3}$**. Density samples are source-specific,
as in the original notebook. Out-of-axis annual values trigger a warning and
remain in the exported CSV; they are not clipped numerically.

The original density notebook's prose said SD >0 and an axis of 0–1000,
but its executable code used the confirmed settings above. That documentation
conflict is resolved here in favor of the user's explicit choice.

## Run instructions

Run all cells in order in the East River HPC environment. The source-tree
lookup supports both HPC path forms. Input tables and generated outputs use
the explicit `OUTPUT_DIR` below; a different configuration-file output path
is reported. Nimbus Sans is required, without silent font substitution.
Both figures export 600-dpi RGB PNG and vector PDF, plus plotting-value CSVs.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import warnings

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

try:
    from PIL import Image
except ImportError as exc:
    raise RuntimeError("Pillow is required to guarantee RGB PNG export.") from exc


# -------------------------------------------------------------------------
# Workflow bootstrap
# -------------------------------------------------------------------------
# Explicit project input/output directory (independent of source-tree alias).
OUTPUT_DIR = Path(
    "/bsuhome/zihaozhang/scratch/iSnobal/"
    "East_River_Workflow_Code_v0.8.2/output/output1"
)
DENSITY_MIN_DEPTH_MM = 100.0
DENSITY_YLIM = (200.0, 400.0)

# Support both HPC path forms used in this project.
REPO_CANDIDATES = [
    Path("/bsuhome/zihaozhang/scratch/iSnobal/East_River_Workflow_Code_v0.8.2"),
    Path("/bsuscratch/zihaozhang/iSnobal/East_River_Workflow_Code_v0.8.2"),
]

REPO_ROOT = next(
    (p for p in REPO_CANDIDATES if (p / "src").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        "Could not find the East River v0.8.2 source tree. Checked:\n"
        + "\n".join(f"  - {p}" for p in REPO_CANDIDATES)
    )

CONFIG_PATH = REPO_ROOT / "config" / "east_river_config.yaml"
sys.path.insert(0, str(REPO_ROOT / "src"))

from east_river_workflow.config import load_config

print("Using workflow root:", REPO_ROOT)


In [ ]:
# -------------------------------------------------------------------------
# East River plotting protocol (JHM / AMS v5)
# -------------------------------------------------------------------------
FONT_NAME = "Nimbus Sans"

available_font_names = {f.name for f in fm.fontManager.ttflist}
if FONT_NAME not in available_font_names:
    raise RuntimeError(
        "Nimbus Sans is required by the East River plotting standard "
        "but is not available in this environment."
    )

resolved_font_path = fm.findfont(
    fm.FontProperties(family=FONT_NAME),
    fallback_to_default=False,
)
print(f"Resolved publication font: {FONT_NAME}")
print(f"Resolved font file: {resolved_font_path}")

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [FONT_NAME],
    "font.size": 18,
    "axes.titlesize": 20,
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
    "font.weight": "normal",
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

MODEL_COLORS = {
    "SNOTEL": "black",
    "iSnobal": "#1F77B4",
    "Noah-MP": "#D62728",
}

MARKER = "o"          # same symbol for both models
MARKERSIZE = 8.5
LINEWIDTH = 2.5

FIGSIZE_BIAS = (21, 19.0)
FIGSIZE_DENSITY = (21, 10.5)
EXPORT_DPI = 600

# Same markers and source line widths as the original density figure.
ROW1_MARKER = MARKER
ROW1_MARKERSIZE = MARKERSIZE
ROW1_LINEWIDTH = {"SNOTEL": 2.0, "iSnobal": 2.5, "Noah-MP": 2.5}


In [ ]:
# -------------------------------------------------------------------------
# Verify the implemented workflow logic for f_SWE_precip
# -------------------------------------------------------------------------
metrics_path = REPO_ROOT / "src" / "east_river_workflow" / "metrics.py"
metrics_text = metrics_path.read_text(encoding="utf-8")

required_fragments = [
    '_complete_precip_to_date(frame, wy, dep[source]["peak_date"], pcol)',
    'peak = dep[source]["peak_swe_mm"]',
    'row[f"f_SWE_precip_{prefix}"] = peak / total',
]

missing_fragments = [frag for frag in required_fragments if frag not in metrics_text]
if missing_fragments:
    raise RuntimeError(
        "Could not verify the expected f_SWE_precip source-code implementation. "
        f"Missing fragments: {missing_fragments}"
    )

metrics_lines = metrics_text.splitlines()
match_idx = next(
    i for i, line in enumerate(metrics_lines)
    if 'row[f"f_SWE_precip_{prefix}"] = peak / total' in line
)

print("Verified metrics.py snippet for f_SWE_precip:")
for j in range(max(0, match_idx - 4), min(len(metrics_lines), match_idx + 3)):
    print(f"{j + 1:4d}: {metrics_lines[j]}")


In [ ]:
# -------------------------------------------------------------------------
# Load workflow outputs
# -------------------------------------------------------------------------
cfg = load_config(CONFIG_PATH)

if Path(cfg.output_dir).resolve() != OUTPUT_DIR.resolve():
    warnings.warn(
        f"Configuration output_dir={cfg.output_dir} differs from the explicit "
        f"project OUTPUT_DIR={OUTPUT_DIR}. Using OUTPUT_DIR for all tables and exports.",
        stacklevel=2,
    )

TABLE_DIR = OUTPUT_DIR / "tables"
PUBLICATION_FIG_DIR = OUTPUT_DIR / "figures" / "publication"
PUBLICATION_TABLE_DIR = OUTPUT_DIR / "tables" / "publication"
PUBLICATION_FIG_DIR.mkdir(parents=True, exist_ok=True)
PUBLICATION_TABLE_DIR.mkdir(parents=True, exist_ok=True)

snotel_path = TABLE_DIR / "snotel_daily.csv"
isnobal_path = TABLE_DIR / "isnobal_station_daily.csv"
noah_path = TABLE_DIR / "noahmp_station_daily.csv"
table4_path = TABLE_DIR / "table4_snotel_annual_metrics.csv"

for path in [snotel_path, isnobal_path, noah_path, table4_path]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required workflow output not found: {path}\n"
            "Run the East River workflow before using this notebook."
        )

snotel = pd.read_csv(snotel_path)
isnobal = pd.read_csv(isnobal_path)
noah = pd.read_csv(noah_path)
table4 = pd.read_csv(table4_path)

for frame in [snotel, isnobal, noah]:
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame["station_id"] = pd.to_numeric(frame["station_id"], errors="raise").astype(int)
    frame["water_year"] = pd.to_numeric(frame["water_year"], errors="raise").astype(int)

station_defs = [
    (int(s["id"]), str(s["name"]))
    for s in cfg.section("snotel")["stations"]
]
configured_stations = {name: station_id for station_id, name in station_defs}
station_order = ["Butte", "Schofield Pass", "Upper Taylor"]
missing_stations = set(station_order) - set(configured_stations)
if missing_stations:
    raise ValueError(f"Required stations missing from configuration: {sorted(missing_stations)}")
station_defs = [(configured_stations[name], name) for name in station_order]
expected_years = list(range(2018, 2027))
station_ids = [station_id for station_id, _ in station_defs]

for frame, source_name in [(snotel, "SNOTEL"), (isnobal, "iSnobal"), (noah, "Noah-MP")]:
    if frame["date"].isna().any():
        raise ValueError(f"{source_name}: invalid or missing daily dates.")
    if frame.duplicated(["station_id", "date"]).any():
        raise ValueError(f"{source_name}: duplicate station/date records.")
    date_water_year = frame["date"].dt.year + (frame["date"].dt.month >= 10).astype(int)
    if not frame["water_year"].eq(date_water_year).all():
        raise ValueError(f"{source_name}: water_year does not match its calendar date.")

for key in ["station_id", "water_year"]:
    table4[key] = pd.to_numeric(table4[key], errors="raise").astype(int)
table4 = table4.loc[
    table4["station_id"].isin(station_ids)
    & table4["water_year"].isin(expected_years)
].copy()
if table4.duplicated(["station_id", "water_year"]).any():
    raise ValueError("Table 4 contains duplicate station/year rows.")
required_keys = {(station_id, wy) for station_id in station_ids for wy in expected_years}
actual_keys = set(zip(table4["station_id"], table4["water_year"]))
if required_keys - actual_keys:
    raise ValueError(f"Table 4 is missing station/year rows: {sorted(required_keys - actual_keys)}")

print("Input/output directory:", OUTPUT_DIR)
print("Stations:", [name for _, name in station_defs])
print("Water years:", expected_years)


## Figure 1 data — precipitation and SWE MB and RMB


In [ ]:
# -------------------------------------------------------------------------
# Calculate annual precipitation and SWE statistics
# -------------------------------------------------------------------------
VARIABLE_SPECS = {
    "precipitation": {
        "SNOTEL": "precip_gapfilled_mm",
        "iSnobal": "precip_nearest_mm",
        "Noah-MP": "precip_mm",
    },
    "SWE": {
        "SNOTEL": "swe_mm",
        "iSnobal": "swe_nearest_mm",
        "Noah-MP": "swe_mm",
    },
}

source_frames = {
    "SNOTEL": snotel,
    "iSnobal": isnobal,
    "Noah-MP": noah,
}

for variable, spec in VARIABLE_SPECS.items():
    for source_name, column in spec.items():
        if column not in source_frames[source_name]:
            raise ValueError(f"{source_name} is missing {variable} column {column}.")
        source_frames[source_name][column] = pd.to_numeric(
            source_frames[source_name][column], errors="coerce"
        )


def mean_bias_and_rmb(reference, model) -> dict[str, float | int]:
    """Pairwise-finite MB and ratio-of-sums RMB; no correlation calculation."""
    reference = np.asarray(reference, dtype=float).ravel()
    model = np.asarray(model, dtype=float).ravel()
    if reference.shape != model.shape:
        raise ValueError("Reference and model arrays must have equal lengths.")
    valid = np.isfinite(reference) & np.isfinite(model)
    reference, model = reference[valid], model[valid]
    n = int(reference.size)
    if n == 0:
        return {"n": 0, "mean_bias": np.nan, "relative_mean_bias_pct": np.nan,
                "reference_sum": np.nan, "model_sum": np.nan}
    difference = model - reference
    reference_sum = float(reference.sum())
    return {
        "n": n,
        "mean_bias": float(difference.mean()),
        "relative_mean_bias_pct": (
            float(100.0 * difference.sum() / reference_sum)
            if reference_sum != 0.0 else np.nan
        ),
        "reference_sum": reference_sum,
        "model_sum": float(model.sum()),
    }


def paired_stats_for_station_year(
    station_id: int,
    water_year: int,
    variable: str,
    model_name: str,
) -> dict[str, float | int]:
    spec = VARIABLE_SPECS[variable]

    ref = (
        source_frames["SNOTEL"]
        .loc[
            source_frames["SNOTEL"]["station_id"].eq(station_id)
            & source_frames["SNOTEL"]["water_year"].eq(water_year),
            ["date", spec["SNOTEL"]],
        ]
        .rename(columns={spec["SNOTEL"]: "reference"})
    )

    mod = (
        source_frames[model_name]
        .loc[
            source_frames[model_name]["station_id"].eq(station_id)
            & source_frames[model_name]["water_year"].eq(water_year),
            ["date", spec[model_name]],
        ]
        .rename(columns={spec[model_name]: "model"})
    )

    paired = ref.merge(
        mod,
        on="date",
        how="inner",
        validate="one_to_one",
    )

    return mean_bias_and_rmb(
        paired["reference"].to_numpy(dtype=float),
        paired["model"].to_numpy(dtype=float),
    )


rows = []

for station_id, station_name in station_defs:
    for wy in expected_years:
        for model_name in ["iSnobal", "Noah-MP"]:
            pstats = paired_stats_for_station_year(
                station_id,
                wy,
                "precipitation",
                model_name,
            )
            sstats = paired_stats_for_station_year(
                station_id,
                wy,
                "SWE",
                model_name,
            )

            rows.append({
                "station_id": station_id,
                "station_name": station_name,
                "water_year": wy,
                "model": model_name,
                "n_precip": int(pstats["n"]),
                "precip_bias_mm_day": float(pstats["mean_bias"]),
                "precip_RMB_pct": float(pstats["relative_mean_bias_pct"]),
                "precip_reference_sum_mm": float(pstats["reference_sum"]),
                "precip_model_sum_mm": float(pstats["model_sum"]),
                "n_SWE": int(sstats["n"]),
                "SWE_RMB_pct": float(sstats["relative_mean_bias_pct"]),
                "SWE_reference_sum_mm": float(sstats["reference_sum"]),
                "SWE_model_sum_mm": float(sstats["model_sum"]),
                "SWE_bias_mm": float(sstats["mean_bias"]),
            })

metrics = (
    pd.DataFrame(rows)
    .sort_values(["station_id", "water_year", "model"])
    .reset_index(drop=True)
)


In [ ]:
# -------------------------------------------------------------------------
# Consistency check: recomputed SWE bias must match validated Table 4
# -------------------------------------------------------------------------
required_table4 = [
    "station_id",
    "water_year",
    "mean_SWE_bias_I_vs_SNOTEL_mm",
    "mean_SWE_bias_N_vs_SNOTEL_mm",
]
missing = [c for c in required_table4 if c not in table4.columns]
if missing:
    raise ValueError(
        "Validated Table 4 is missing required SWE-bias columns: "
        + ", ".join(missing)
    )

check_rows = []

for model_name, table4_col in [
    ("iSnobal", "mean_SWE_bias_I_vs_SNOTEL_mm"),
    ("Noah-MP", "mean_SWE_bias_N_vs_SNOTEL_mm"),
]:
    calc = metrics.loc[
        metrics["model"].eq(model_name),
        ["station_id", "water_year", "SWE_bias_mm"],
    ]

    expected = table4[
        ["station_id", "water_year", table4_col]
    ].rename(columns={table4_col: "workflow_SWE_bias_mm"})

    joined = calc.merge(
        expected,
        on=["station_id", "water_year"],
        how="left",
        validate="one_to_one",
    )
    joined["difference_mm"] = (
        joined["SWE_bias_mm"] - joined["workflow_SWE_bias_mm"]
    )
    joined["model"] = model_name
    check_rows.append(joined)

swe_bias_check = pd.concat(check_rows, ignore_index=True)

matches = np.isclose(
    swe_bias_check["SWE_bias_mm"].to_numpy(dtype=float),
    swe_bias_check["workflow_SWE_bias_mm"].to_numpy(dtype=float),
    atol=1e-9, rtol=0.0, equal_nan=True,
)
if not matches.all():
    conflicts = swe_bias_check.loc[~matches]
    display(conflicts)
    raise RuntimeError(
        "Recomputed SWE MB conflicts with Table 4. Conflicting rows are shown above; "
        "resolve the source discrepancy before exporting or plotting."
    )
finite_diff = swe_bias_check["difference_mm"].dropna().abs()
max_abs_diff = float(finite_diff.max()) if len(finite_diff) else np.nan
print("SWE-bias consistency check: PASSED")
print("Maximum finite absolute difference (mm):", max_abs_diff)

metrics_csv = PUBLICATION_TABLE_DIR / "station_year_precip_swe_MB_RMB_wy2018_2026.csv"
metrics.to_csv(metrics_csv, index=False)
print("Saved metrics CSV:", metrics_csv)
display(metrics.head(18))


In [ ]:
# -------------------------------------------------------------------------
# 4 × 3 figure
# -------------------------------------------------------------------------
def style_standard_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(axis="both", which="major", width=1.0, length=5)


def nice_symmetric_limit(values, pad_fraction=0.10):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if not len(arr):
        return 1.0

    target = np.max(np.abs(arr)) * (1.0 + pad_fraction)
    if target == 0:
        return 1.0

    magnitude = 10.0 ** np.floor(np.log10(target))
    scaled = target / magnitude

    for candidate in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 7.5, 10.0]:
        if scaled <= candidate:
            return candidate * magnitude

    return 10.0 * magnitude


# Shared row limits
precip_bias_lim = nice_symmetric_limit(metrics["precip_bias_mm_day"])
swe_bias_lim = nice_symmetric_limit(metrics["SWE_bias_mm"])

precip_rmb_lim = nice_symmetric_limit(metrics["precip_RMB_pct"])
swe_rmb_lim = nice_symmetric_limit(metrics["SWE_RMB_pct"])

ROW_SPECS = [
    {
        "column": "precip_bias_mm_day",
        "ylabel": r"Precip MB (mm day$^{-1}$)",
        "ylim": (-precip_bias_lim, precip_bias_lim),
        "zero_line": True,
    },
    {
        "column": "precip_RMB_pct",
        "ylabel": "Precip RMB (%)",
        "ylim": (-precip_rmb_lim, precip_rmb_lim),
        "zero_line": True,
    },
    {
        "column": "SWE_bias_mm",
        "ylabel": "SWE MB (mm)",
        "ylim": (-swe_bias_lim, swe_bias_lim),
        "zero_line": True,
    },
    {
        "column": "SWE_RMB_pct",
        "ylabel": "SWE RMB (%)",
        "ylim": (-swe_rmb_lim, swe_rmb_lim),
        "zero_line": True,
    },
]


fig, axes = plt.subplots(
    4,
    3,
    figsize=FIGSIZE_BIAS,
    sharex=True,
    sharey="row",
    squeeze=False,
)

for row_idx, row_spec in enumerate(ROW_SPECS):
    metric_col = row_spec["column"]

    for col_idx, (station_id, station_name) in enumerate(station_defs):
        ax = axes[row_idx, col_idx]

        ss = metrics[
            metrics["station_id"].eq(station_id)
        ].copy()

        for model_name in ["iSnobal", "Noah-MP"]:
            ds = (
                ss[ss["model"].eq(model_name)]
                .sort_values("water_year")
            )

            ax.plot(
                ds["water_year"],
                ds[metric_col],
                color=MODEL_COLORS[model_name],
                linewidth=LINEWIDTH,
                marker=MARKER,
                markersize=MARKERSIZE,
                markerfacecolor=MODEL_COLORS[model_name],
                markeredgecolor="black",
                markeredgewidth=0.9,
                label=model_name,
                zorder=4,
            )

        if row_spec["zero_line"]:
            ax.axhline(
                0,
                color="0.55",
                linewidth=1.0,
                linestyle=(0, (2.5, 2.5)),
                zorder=1,
            )

        ax.set_xlim(expected_years[0] - 0.4, expected_years[-1] + 0.4)
        ax.set_xticks(expected_years)
        ax.set_ylim(*row_spec["ylim"])
        style_standard_axis(ax)

        if row_idx == 0:
            ax.set_title(
                station_name,
                fontsize=20,
                fontweight="normal",
                pad=8,
            )

        if row_idx < 3:
            ax.tick_params(labelbottom=False)
        else:
            ax.set_xticklabels(
                [str(wy) for wy in expected_years],
                rotation=45,
                ha="right",
            )

    # One y label per row, attached to the left subplot.
    axes[row_idx, 0].set_ylabel(
        row_spec["ylabel"],
        fontsize=18,
        labelpad=12,
    )


legend_handles = [
    Line2D(
        [],
        [],
        linestyle="-",
        linewidth=LINEWIDTH,
        color=MODEL_COLORS[model_name],
        marker=MARKER,
        markersize=MARKERSIZE,
        markerfacecolor=MODEL_COLORS[model_name],
        markeredgecolor="black",
        markeredgewidth=0.8,
        label=model_name,
    )
    for model_name in ["iSnobal", "Noah-MP"]
]

axes[0, 0].legend(
    handles=legend_handles,
    loc="upper right",
    ncol=1,
    frameon=True,
    facecolor="white",
    edgecolor="0.4",
    framealpha=0.9,
    fancybox=False,
    borderpad=0.5,
    handlelength=2.2,
)

for i, ax in enumerate(axes.flat):
    ax.text(
        0.02, 0.97,
        f"({chr(97 + i)})",
        transform=ax.transAxes,
        fontsize=18,
        fontweight="normal",
        ha="left",
        va="top",
        zorder=10,
    )

fig.subplots_adjust(
    top=0.965,
    bottom=0.10,
    left=0.09,
    right=0.985,
    hspace=0.18,
    wspace=0.12,
)

figure_stem = "station_precip_swe_MB_RMB_all_years_4x3"
png_path = PUBLICATION_FIG_DIR / f"{figure_stem}.png"
pdf_path = PUBLICATION_FIG_DIR / f"{figure_stem}.pdf"

fig.savefig(
    png_path,
    dpi=EXPORT_DPI,
    bbox_inches="tight",
    pad_inches=0.05,
    facecolor="white",
    transparent=False,
)
fig.savefig(
    pdf_path,
    bbox_inches="tight",
    pad_inches=0.05,
    facecolor="white",
    transparent=False,
)

with Image.open(png_path) as im:
    if im.mode != "RGB":
        rgb = im.convert("RGB")
        rgb.save(
            png_path,
            dpi=(EXPORT_DPI, EXPORT_DPI),
        )

print("Saved figure:", png_path)
print("Saved figure:", pdf_path)

plt.show()
plt.close(fig)


## Figure 2 data — fSWE,precip and annual-mean snow density

The confirmed density filter and axis limits are retained. This is a separate 2 × 3 figure.


In [ ]:
# -------------------------------------------------------------------------
# Row 1 data: validated annual f_SWE_precip values
# -------------------------------------------------------------------------

required_table4 = [
    "station_id", "station_name", "water_year",
    "f_SWE_precip_SNOTEL", "f_SWE_precip_I", "f_SWE_precip_N",
]
missing = [c for c in required_table4 if c not in table4.columns]
if missing:
    raise ValueError(
        "Table 4 is missing required f_SWE_precip fields: " + ", ".join(missing)
    )

row1 = table4[required_table4].copy()
row1["station_id"] = pd.to_numeric(row1["station_id"], errors="raise").astype(int)
row1["water_year"] = pd.to_numeric(row1["water_year"], errors="raise").astype(int)

row1 = row1.rename(columns={
    "f_SWE_precip_SNOTEL": "SNOTEL",
    "f_SWE_precip_I": "iSnobal",
    "f_SWE_precip_N": "Noah-MP",
})

row1_long = row1.melt(
    id_vars=["station_id", "station_name", "water_year"],
    value_vars=["SNOTEL", "iSnobal", "Noah-MP"],
    var_name="source",
    value_name="f_SWE_precip",
).sort_values(["station_id", "water_year", "source"]).reset_index(drop=True)

row1_csv = PUBLICATION_TABLE_DIR / "station_year_fSWE_precip_values_wy2018_2026.csv"
row1_long.to_csv(row1_csv, index=False)
print("Saved row-1 CSV:", row1_csv)

display(row1_long.head(12))


In [ ]:
# -------------------------------------------------------------------------
# Row 2 data: annual-mean snow density from daily values
# -------------------------------------------------------------------------
# The three daily tables do not use identical snow-state names.
SOURCE_DAILY_COLUMNS = {
    "SNOTEL": {
        "swe": "swe_mm",
        "depth": "snow_depth_mm",
    },
    "iSnobal": {
        "swe": "swe_nearest_mm",
        "depth": "snow_depth_nearest_mm",
    },
    "Noah-MP": {
        "swe": "swe_mm",
        "depth": "snow_depth_mm",
    },
}


def load_daily_source(frame: pd.DataFrame, source_name: str) -> pd.DataFrame:
    frame = frame.copy()

    swe_col = SOURCE_DAILY_COLUMNS[source_name]["swe"]
    depth_col = SOURCE_DAILY_COLUMNS[source_name]["depth"]

    required = [
        "date",
        "station_id",
        "station_name",
        "water_year",
        swe_col,
        depth_col,
    ]
    missing = [c for c in required if c not in frame.columns]
    if missing:
        raise ValueError(
            f"{source_name} daily table is missing required columns: {missing}\n"
            f"Available columns: {list(frame.columns)}"
        )

    out = frame[
        ["date", "station_id", "station_name", "water_year", swe_col, depth_col]
    ].copy()

    out = out.rename(columns={
        swe_col: "swe_mm",
        depth_col: "snow_depth_mm",
    })

    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["station_id"] = pd.to_numeric(out["station_id"], errors="raise").astype(int)
    out["water_year"] = pd.to_numeric(out["water_year"], errors="raise").astype(int)
    out["swe_mm"] = pd.to_numeric(out["swe_mm"], errors="coerce")
    out["snow_depth_mm"] = pd.to_numeric(out["snow_depth_mm"], errors="coerce")
    out["source"] = source_name

    return out


daily_frames = [
    load_daily_source(snotel, "SNOTEL"),
    load_daily_source(isnobal, "iSnobal"),
    load_daily_source(noah, "Noah-MP"),
]

density_daily = pd.concat(daily_frames, ignore_index=True)
density_daily = density_daily[density_daily["water_year"].isin(expected_years) & density_daily["station_id"].isin(station_ids)].copy()

valid = (
    np.isfinite(density_daily["swe_mm"])
    & np.isfinite(density_daily["snow_depth_mm"])
    & (density_daily["snow_depth_mm"] >= DENSITY_MIN_DEPTH_MM)
)
density_daily = density_daily.loc[valid].copy()

density_daily["snow_density_kg_m3"] = (
    1000.0 * density_daily["swe_mm"] / density_daily["snow_depth_mm"]
)

# User-approved annual reduction for row 2: annual mean of valid daily density.
density_annual = (
    density_daily.groupby(
        ["station_id", "station_name", "water_year", "source"],
        as_index=False,
    )["snow_density_kg_m3"]
    .mean()
    .rename(columns={"snow_density_kg_m3": "annual_mean_snow_density_kg_m3"})
    .sort_values(["station_id", "water_year", "source"])
    .reset_index(drop=True)
)

density_csv = PUBLICATION_TABLE_DIR / "annual_mean_snow_density_values_wy2018_2026.csv"
density_annual.to_csv(
    density_csv,
    index=False,
)
print("Saved row-2 CSV:", density_csv)

display(density_annual.head(12))

# Preserve empty station/year/source combinations as gaps rather than joining across them.
density_grid = pd.MultiIndex.from_product(
    [station_ids, expected_years, ["SNOTEL", "iSnobal", "Noah-MP"]],
    names=["station_id", "water_year", "source"],
).to_frame(index=False)
density_annual = density_grid.merge(
    density_annual.drop(columns="station_name"),
    on=["station_id", "water_year", "source"], how="left", validate="one_to_one",
)
density_annual.insert(1, "station_name", density_annual["station_id"].map(dict(station_defs)))
density_annual.to_csv(density_csv, index=False)
outside_axis = density_annual.loc[
    density_annual["annual_mean_snow_density_kg_m3"].notna()
    & ~density_annual["annual_mean_snow_density_kg_m3"].between(*DENSITY_YLIM)
]
if not outside_axis.empty:
    warnings.warn(
        f"{len(outside_axis)} annual density values fall outside the confirmed "
        f"{DENSITY_YLIM} axis. Their original values remain in the CSV.",
        stacklevel=2,
    )
    display(outside_axis)


In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=FIGSIZE_DENSITY,
    squeeze=False,
    sharex=False,
    sharey="row",
)

# -----------------------
# Row 1: fSWE,precip
# -----------------------
for col, (station_id, station_name) in enumerate(station_defs):
    ax = axes[0, col]
    ss = row1_long[row1_long["station_id"].eq(station_id)].copy()

    for source in ["SNOTEL", "iSnobal", "Noah-MP"]:
        ds = ss[ss["source"].eq(source)].sort_values("water_year")
        ax.plot(
            ds["water_year"],
            ds["f_SWE_precip"],
            color=MODEL_COLORS[source],
            linewidth=ROW1_LINEWIDTH[source],
            marker=ROW1_MARKER,
            markersize=ROW1_MARKERSIZE,
            markerfacecolor=MODEL_COLORS[source],
            markeredgecolor="black",
            markeredgewidth=0.9,
            label=source,
            zorder=4,
        )

    ax.set_title(station_name, fontsize=20, fontweight="normal", pad=8)
    ax.set_xlim(expected_years[0] - 0.4, expected_years[-1] + 0.4)
    ax.set_xticks(expected_years)
    ax.set_xticklabels([])
    style_standard_axis(ax)

# -----------------------
# Row 2: annual-mean snow density
# -----------------------
for col, (station_id, station_name) in enumerate(station_defs):
    ax = axes[1, col]
    ss = density_annual[density_annual["station_id"].eq(station_id)].copy()

    for source in ["SNOTEL", "iSnobal", "Noah-MP"]:
        ds = ss[ss["source"].eq(source)].sort_values("water_year")
        ax.plot(
            ds["water_year"],
            ds["annual_mean_snow_density_kg_m3"],
            color=MODEL_COLORS[source],
            linewidth=ROW1_LINEWIDTH[source],
            marker=ROW1_MARKER,
            markersize=ROW1_MARKERSIZE,
            markerfacecolor=MODEL_COLORS[source],
            markeredgecolor="black",
            markeredgewidth=0.9,
            label=source,
            zorder=4,
        )

    ax.set_xlim(expected_years[0] - 0.4, expected_years[-1] + 0.4)
    ax.set_xticks(expected_years)
    ax.set_xticklabels([str(wy) for wy in expected_years], rotation=45, ha="right")
    ax.set_ylim(*DENSITY_YLIM)
    style_standard_axis(ax)

# Figure-level labels
fig.text(
    0.015,
    0.75,
    r"$f_{\mathrm{SWE,precip}}$",
    rotation=90,
    ha="center",
    va="center",
)
fig.text(
    0.015,
    0.32,
    r"Snow density (kg m$^{-3}$)",
    rotation=90,
    ha="center",
    va="center",
)
#fig.supxlabel("Water year", y=0.045)

# Top-row legend
top_legend_handles = [
    Line2D(
        [], [],
        linestyle="-",
        linewidth=ROW1_LINEWIDTH[source],
        color=MODEL_COLORS[source],
        marker=ROW1_MARKER,
        markersize=ROW1_MARKERSIZE,
        markerfacecolor=MODEL_COLORS[source],
        markeredgecolor="black",
        markeredgewidth=0.8,
        label=source,
    )
    for source in ["SNOTEL", "iSnobal", "Noah-MP"]
]
axes[0, 0].legend(
    handles=top_legend_handles,
    loc="upper right",
    ncol=1,
    frameon=True,
    facecolor="white",
    edgecolor="0.4",
    framealpha=0.9,
    fancybox=False,
    borderpad=0.5,
    handlelength=2.2,
)
for i, ax in enumerate(axes.flat):
    ax.text(
        0.02, 0.97,
        f"({chr(97 + i)})",
        transform=ax.transAxes,
        fontsize=18,
        fontweight="normal",
        ha="left",
        va="top",
        zorder=10,
    )


fig.subplots_adjust(
    top=0.94,
    bottom=0.12,
    left=0.07,
    right=0.985,
    hspace=0.18,
    wspace=0.12,
)

figure_stem = "station_fSWE_precip_and_mean_snow_density_2x3"
png_path = PUBLICATION_FIG_DIR / f"{figure_stem}.png"
pdf_path = PUBLICATION_FIG_DIR / f"{figure_stem}.pdf"

fig.savefig(
    png_path,
    dpi=EXPORT_DPI,
    bbox_inches="tight",
    pad_inches=0.05,
    facecolor="white",
    transparent=False,
)
fig.savefig(
    pdf_path,
    bbox_inches="tight",
    pad_inches=0.05,
    facecolor="white",
    transparent=False,
)

with Image.open(png_path) as im:
    if im.mode != "RGB":
        rgb = im.convert("RGB")
        rgb.save(png_path, dpi=(EXPORT_DPI, EXPORT_DPI))

print("Saved figure:", png_path)
print("Saved figure:", pdf_path)

plt.show()
plt.close(fig)


## Generated outputs

Under `OUTPUT_DIR / "figures/publication"`:
- `station_precip_swe_MB_RMB_all_years_4x3.png` and `.pdf`
- `station_fSWE_precip_and_mean_snow_density_2x3.png` and `.pdf`

Under `OUTPUT_DIR / "tables/publication"`:
- `station_year_precip_swe_MB_RMB_wy2018_2026.csv`
- `station_year_fSWE_precip_values_wy2018_2026.csv`
- `annual_mean_snow_density_values_wy2018_2026.csv`

The MB/RMB CSV contains paired sample counts, MB, RMB (%), and paired
reference/model sums for both variables. The SWE sums are sums of daily SWE
states used to audit the RMB denominator, **not** a precipitation total or
an independent water-balance metric. All CSVs retain the computed numeric
precision. Missing or undefined statistics appear as empty CSV fields.

The fraction and density output filenames are retained from the original
notebook; rerunning this notebook updates those outputs. The MB/RMB figure
and CSV use new filenames. No old correlation results are carried into this notebook.
